# Validation-selected SVM–neural fusion
Runs the next leakage-aware fusion experiment.

## Goal

For each outer fold:

1. recreate the **same RBF-SVM pipeline** used in the original classical-ML baseline;
2. produce SVM probabilities for the **outer validation** participants;
3. combine those validation probabilities with the medium neural model's validation probabilities;
4. select a continuous fusion weight

\[
p_{fusion} = \alpha p_{SVM} + (1-\alpha)p_{neural}
\]

using **validation ROC AUC only**;
5. lock that fold-specific `alpha`;
6. apply it once to the corresponding held-out outer test fold.

Two neural outputs are tested separately:

- medium **Hybrid**
- medium **TMC**

The main `alpha` grid is coarse and pre-specified: `0.00, 0.05, ..., 1.00`.

## Why the SVM is refit here

The downloaded SVM ZIP contains test predictions but not validation probabilities. The original SVM models were not saved. Therefore this notebook reconstructs the original SVM training/search procedure from the baseline notebook:

- same 25 continuous + 3 categorical predictors;
- same missingness masks and branch masks;
- same 64-D fold-specific MRI embeddings;
- same 32-config RBF-SVM grid;
- same 5-fold inner CV with search seed 17;
- same top-5 candidate validation selection;
- same final seeds 17, 42, 73.

The notebook compares the newly reproduced SVM test probabilities against the downloaded original SVM test probabilities as a sanity check.

## Important interpretation

This experiment is still **post-hoc exploratory** because earlier outer-test results motivated the fusion experiment. The test fold is not used to select `alpha`, but the overall research direction was informed by prior test inspection. Treat the result as evidence for what should be externally validated (e.g. on AIBL), not as a pristine confirmatory estimate.


In [ ]:
from pathlib import Path
import json
import ast
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)

# Paths

DATA_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")
RUNS_ROOT = Path("/home/jovyan/thesis_runs")

NEURAL_ROOT = (
    RUNS_ROOT
    / "gated_cmt_fixed_equal_fusion_medium_v1_md050"
)

ORIGINAL_SVM_TEST_DIR = RUNS_ROOT / "svm_rbf_predictions"

OUTPUT_ROOT = (
    RUNS_ROOT
    / "validation_selected_svm_neural_fusion_v1"
)
TABLES_DIR = OUTPUT_ROOT / "tables"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
FIGURES_DIR = OUTPUT_ROOT / "figures"

for d in [OUTPUT_ROOT, TABLES_DIR, PREDICTIONS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Experiment contract

FOLDS = [0, 1, 2, 3, 4]
SEEDS = [17, 42, 73]
SEARCH_SEED = 17

INNER_CV_SPLITS = 5
TOP_K_VALIDATION = 5

ALPHA_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2)

SVM_C_VALUES = [0.1, 1.0, 10.0, 100.0]
SVM_GAMMA_VALUES = ["scale", 0.001, 0.01, 0.1]

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20261004

print("Data root:", DATA_ROOT)
print("Neural root:", NEURAL_ROOT)
print("Original SVM test predictions:", ORIGINAL_SVM_TEST_DIR)
print("Output root:", OUTPUT_ROOT)
print("Alpha grid:", ALPHA_GRID.tolist())


## 1. Reproduce the baseline predictor contract


In [ ]:
RID_COL = "RID"
TARGET_COL = "MCI_PROGNOSIS_TARGET"
ROLE_COL = "DATA_ROLE"
OUTER_FOLD_COL = "OUTER_FOLD"

CONTINUOUS_COLS = [
    "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
    "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
    "ADAS__TOTSCORE__Z",
    "ADAS__TOTAL13__Z",
    "MMSE__MMSE_TOTAL_SCORE__Z",
    "MMSE__MMSE_ORIENTATION_SCORE__Z",
    "MMSE__MMSE_REGISTRATION_SCORE__Z",
    "MMSE__MMSE_ATTENTION_SCORE__Z",
    "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
    "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
    "FAQ__FAQTOTAL__Z",
    "CSF__ABETA40__Z",
    "CSF__ABETA42__Z",
    "CSF__TAU__Z",
    "CSF__PTAU__Z",
    "CSF__ABETA42_40_RATIO__Z",
    "PLASMA__pT217_F__Z",
    "PLASMA__AB42_F__Z",
    "PLASMA__AB40_F__Z",
    "PLASMA__AB42_AB40_F__Z",
    "PLASMA__pT217_AB42_F__Z",
    "PLASMA__NfL_Q__Z",
    "PLASMA__GFAP_Q__Z",
    "PLASMA__NfL_F__Z",
    "PLASMA__GFAP_F__Z",
]

CATEGORICAL_COLS = [
    "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
    "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
    "APOE__APOE4_ALLELE_COUNT__IDX",
]

BRANCH_MASK_COLS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
]

MRI_BRANCH_MASK_COL = "BRANCH_MASK__MRI"

MRI_EMBEDDING_DIM = 64
MRI_EMBEDDING_COLS = [
    f"MRI_EMB_{i:02d}"
    for i in range(MRI_EMBEDDING_DIM)
]

TABULAR_PREDICTOR_COLS = CONTINUOUS_COLS + CATEGORICAL_COLS

def predictor_to_feature_mask(predictor):
    base = predictor
    for suffix in ("__Z", "__IDX"):
        if base.endswith(suffix):
            base = base[:-len(suffix)]
            break
    return "FEATURE_MASK__" + base.replace("__", "_")

EXPECTED_FEATURE_MASKS = [
    predictor_to_feature_mask(c)
    for c in TABULAR_PREDICTOR_COLS
]

assert len(CONTINUOUS_COLS) == 25
assert len(CATEGORICAL_COLS) == 3
assert len(EXPECTED_FEATURE_MASKS) == 28

print("Continuous predictors:", len(CONTINUOUS_COLS))
print("Categorical predictors:", len(CATEGORICAL_COLS))
print("Feature masks:", len(EXPECTED_FEATURE_MASKS))
print("MRI embedding dimensions:", MRI_EMBEDDING_DIM)


## 2. Auto-discover the prepared fold tables and MRI embeddings

The original Google Drive locations are listed below for reference:

- prepared categorical-encoded folds:  
  `MyDrive/adni_mri/models/3mt_tmc_evidential/fold_ready_inputs_categorical_encoded/mci_prognosis/`
- MRI embeddings:  
  `MyDrive/adni_mri/models/3mt_tmc_evidential/standard_ml_baselines/mri_embeddings/`

On JHub, the notebook accepts either the old categorical-encoded filenames **or** the newer `final_task_ready` fold files if they contain the same required predictor contract.

If the five MRI embedding CSVs are missing, the input check prints the required filenames. Only those CSVs need to be copied; MRI volumes and model retraining are not required.


In [ ]:
# Optional manual overrides. Leave as None for auto-discovery.
FOLD_INPUT_ROOT_OVERRIDE = None
MRI_EMBEDDING_ROOT_OVERRIDE = None

fold_root_candidates = [
    DATA_ROOT
    / "models"
    / "3mt_tmc_evidential"
    / "fold_ready_inputs_categorical_encoded"
    / "mci_prognosis",

    DATA_ROOT
    / "models"
    / "fold_ready_inputs_categorical_encoded"
    / "mci_prognosis",

    DATA_ROOT
    / "models"
    / "final_task_ready_inputs"
    / "mci_prognosis",
]

mri_root_candidates = [
    DATA_ROOT
    / "models"
    / "3mt_tmc_evidential"
    / "standard_ml_baselines"
    / "mri_embeddings",

    DATA_ROOT
    / "models"
    / "standard_ml_baselines"
    / "mri_embeddings",

    RUNS_ROOT
    / "standard_ml_baselines"
    / "mri_embeddings",

    RUNS_ROOT
    / "svm_rbf_predictions"
    / "mri_embeddings",
]

def fold_file_in_root(root, fold):
    candidates = [
        root
        / f"mci_prognosis_outer_fold_{fold}_categorical_encoded.csv",

        root
        / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None

if FOLD_INPUT_ROOT_OVERRIDE is not None:
    FOLD_INPUT_ROOT = Path(FOLD_INPUT_ROOT_OVERRIDE)
else:
    FOLD_INPUT_ROOT = None
    for candidate in fold_root_candidates:
        if fold_file_in_root(candidate, 0) is not None:
            FOLD_INPUT_ROOT = candidate
            break

if MRI_EMBEDDING_ROOT_OVERRIDE is not None:
    MRI_EMBEDDING_ROOT = Path(MRI_EMBEDDING_ROOT_OVERRIDE)
else:
    MRI_EMBEDDING_ROOT = None
    for candidate in mri_root_candidates:
        probe = (
            candidate
            / "mci_prognosis_fold_0_seed_17_mri_embeddings.csv"
        )
        if probe.exists():
            MRI_EMBEDDING_ROOT = candidate
            break

print("Discovered fold input root:", FOLD_INPUT_ROOT)
print("Discovered MRI embedding root:", MRI_EMBEDDING_ROOT)


## 3. Input check


In [ ]:
missing_messages = []

if FOLD_INPUT_ROOT is None:
    missing_messages.append(
        "Could not find prepared fold CSVs."
    )
else:
    for fold in FOLDS:
        path = fold_file_in_root(FOLD_INPUT_ROOT, fold)
        if path is None:
            missing_messages.append(
                f"Missing fold input for fold {fold} under {FOLD_INPUT_ROOT}"
            )

if MRI_EMBEDDING_ROOT is None:
    missing_messages.append(
        "Could not find the MRI embedding directory."
    )
else:
    for fold in FOLDS:
        path = (
            MRI_EMBEDDING_ROOT
            / f"mci_prognosis_fold_{fold}_seed_17_mri_embeddings.csv"
        )
        if not path.exists():
            missing_messages.append(
                f"Missing MRI embedding: {path}"
            )

for fold in FOLDS:
    for seed in SEEDS:
        path = (
            ORIGINAL_SVM_TEST_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )
        if not path.exists():
            missing_messages.append(
                f"Missing downloaded original SVM prediction: {path}"
            )

        for filename in [
            "best_validation_predictions.csv",
            "test_predictions.csv",
        ]:
            neural_path = (
                NEURAL_ROOT
                / f"seed_{seed}"
                / "mci_prognosis"
                / f"fold_{fold}"
                / "predictions"
                / filename
            )
            if not neural_path.exists():
                missing_messages.append(
                    f"Missing neural prediction: {neural_path}"
                )

if missing_messages:
    print("PRE-FLIGHT FAILED\n")
    for msg in missing_messages:
        print(" -", msg)

    if MRI_EMBEDDING_ROOT is None:
        print("\nIf MRI embeddings are the missing item, download these FIVE files from:")
        print(
            "Google Drive -> MyDrive/adni_mri/models/3mt_tmc_evidential/"
            "standard_ml_baselines/mri_embeddings/"
        )
        for fold in FOLDS:
            print(
                "   "
                f"mci_prognosis_fold_{fold}_seed_17_mri_embeddings.csv"
            )
        print(
            "\nUpload them into e.g. "
            "/home/jovyan/thesis_runs/svm_rbf_predictions/mri_embeddings/"
        )
        print(
            "Then restart this notebook. Auto-discovery will find them."
        )

    raise FileNotFoundError(
        f"{len(missing_messages)} required item(s) are missing."
    )

print("Input check passed.")
print("Fold inputs:", FOLD_INPUT_ROOT)
print("MRI embeddings:", MRI_EMBEDDING_ROOT)


## 4. Load and preprocess folds exactly like the original SVM baseline


In [ ]:
def _coerce_binary_mask(series, name):
    out = pd.to_numeric(
        series,
        errors="raise",
    ).astype(int)

    bad = set(out.unique()) - {0, 1}

    if bad:
        raise ValueError(
            f"{name} contains values outside 0/1: {sorted(bad)}"
        )

    return out

def load_and_validate_fold(fold):
    path = fold_file_in_root(FOLD_INPUT_ROOT, fold)

    if path is None:
        raise FileNotFoundError(
            f"No prepared fold file found for fold {fold}."
        )

    df = pd.read_csv(path)

    # Support newer final-task files whose target may be MODEL_TARGET.
    if TARGET_COL not in df.columns:
        if "MODEL_TARGET" in df.columns:
            df[TARGET_COL] = df["MODEL_TARGET"].astype(int)
        else:
            raise KeyError(
                f"{path}: neither {TARGET_COL} nor MODEL_TARGET exists."
            )

    # Support final-task files that may not carry OUTER_FOLD explicitly.
    if OUTER_FOLD_COL not in df.columns:
        df[OUTER_FOLD_COL] = fold

    required = {
        RID_COL,
        TARGET_COL,
        ROLE_COL,
        OUTER_FOLD_COL,
        *TABULAR_PREDICTOR_COLS,
        *EXPECTED_FEATURE_MASKS,
        *BRANCH_MASK_COLS,
        MRI_BRANCH_MASK_COL,
    }

    missing = sorted(
        required - set(df.columns)
    )

    if missing:
        raise ValueError(
            f"Fold {fold}: required columns missing from {path}:\n{missing}"
        )

    if len(df) != 544:
        raise ValueError(
            f"Fold {fold}: expected 544 rows, found {len(df)}"
        )

    if df[RID_COL].duplicated().any():
        raise ValueError(
            f"Fold {fold}: duplicate RID values."
        )

    for col in EXPECTED_FEATURE_MASKS + BRANCH_MASK_COLS + [MRI_BRANCH_MASK_COL]:
        df[col] = _coerce_binary_mask(
            df[col],
            col,
        )

    for predictor in CONTINUOUS_COLS:
        mask_col = predictor_to_feature_mask(
            predictor
        )

        numeric = pd.to_numeric(
            df[predictor],
            errors="coerce",
        )

        observed_but_nan = (
            (df[mask_col] == 1)
            & numeric.isna()
        )

        if observed_but_nan.any():
            rids = (
                df.loc[
                    observed_but_nan,
                    RID_COL,
                ]
                .head()
                .tolist()
            )

            raise ValueError(
                f"Fold {fold}: {predictor} is observed by mask "
                f"but NaN for RIDs {rids}"
            )

        numeric = numeric.fillna(0.0)
        numeric.loc[
            df[mask_col] == 0
        ] = 0.0

        df[predictor] = numeric

    for predictor in CATEGORICAL_COLS:
        mask_col = predictor_to_feature_mask(
            predictor
        )

        df[predictor] = pd.to_numeric(
            df[predictor],
            errors="raise",
        ).astype(int)

        df.loc[
            df[mask_col] == 0,
            predictor,
        ] = 0

    return df

def load_mri_embeddings(fold, fold_df):
    path = (
        MRI_EMBEDDING_ROOT
        / f"mci_prognosis_fold_{fold}_seed_17_mri_embeddings.csv"
    )

    emb = pd.read_csv(path)

    required = {
        RID_COL,
        MRI_BRANCH_MASK_COL,
        *MRI_EMBEDDING_COLS,
    }

    missing = sorted(
        required - set(emb.columns)
    )

    if missing:
        raise ValueError(
            f"{path}: missing MRI embedding columns: {missing}"
        )

    if len(emb) != 544:
        raise ValueError(
            f"Fold {fold}: expected 544 embedding rows, got {len(emb)}"
        )

    emb[RID_COL] = pd.to_numeric(
        emb[RID_COL],
        errors="raise",
    ).astype(int)

    if emb[RID_COL].duplicated().any():
        raise ValueError(
            f"Fold {fold}: duplicate RID in MRI embeddings."
        )

    for col in MRI_EMBEDDING_COLS:
        emb[col] = pd.to_numeric(
            emb[col],
            errors="raise",
        ).astype(float)

    if not np.isfinite(
        emb[MRI_EMBEDDING_COLS].to_numpy()
    ).all():
        raise ValueError(
            f"Fold {fold}: non-finite MRI embedding values."
        )

    base_ids = set(
        fold_df[RID_COL].astype(int)
    )

    emb_ids = set(
        emb[RID_COL].astype(int)
    )

    if base_ids != emb_ids:
        raise ValueError(
            f"Fold {fold}: RID mismatch between fold table and MRI embeddings."
        )

    return emb[
        [RID_COL, *MRI_EMBEDDING_COLS]
    ].copy()

def make_outer_fold_matrices(fold):
    df = load_and_validate_fold(fold)

    emb = load_mri_embeddings(
        fold,
        df,
    )

    df = df.merge(
        emb,
        on=RID_COL,
        how="left",
        validate="one_to_one",
    )

    train = df.loc[
        df[ROLE_COL] == "train"
    ].copy()

    val = df.loc[
        df[ROLE_COL] == "validation"
    ].copy()

    test = df.loc[
        df[ROLE_COL] == "test"
    ].copy()

    try:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float64,
        )
    except TypeError:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float64,
        )

    ohe.fit(
        train[CATEGORICAL_COLS]
    )

    def transform(part):
        x_cont = part[
            CONTINUOUS_COLS
        ].to_numpy(dtype=np.float64)

        x_cat = ohe.transform(
            part[CATEGORICAL_COLS]
        )

        x_mri = part[
            MRI_EMBEDDING_COLS
        ].to_numpy(dtype=np.float64)

        extra_cols = (
            EXPECTED_FEATURE_MASKS
            + BRANCH_MASK_COLS
            + [MRI_BRANCH_MASK_COL]
        )

        x_extra = part[
            extra_cols
        ].to_numpy(dtype=np.float64)

        X = np.concatenate(
            [
                x_cont,
                x_cat,
                x_mri,
                x_extra,
            ],
            axis=1,
        )

        y = part[
            TARGET_COL
        ].to_numpy(dtype=int)

        rid = part[
            RID_COL
        ].to_numpy(dtype=int)

        return X, y, rid

    X_train, y_train, rid_train = transform(train)
    X_val, y_val, rid_val = transform(val)
    X_test, y_test, rid_test = transform(test)

    return {
        "X_train": X_train,
        "y_train": y_train,
        "rid_train": rid_train,
        "X_val": X_val,
        "y_val": y_val,
        "rid_val": rid_val,
        "X_test": X_test,
        "y_test": y_test,
        "rid_test": rid_test,
    }

example = make_outer_fold_matrices(0)

print("Fold 0 train:", example["X_train"].shape)
print("Fold 0 validation:", example["X_val"].shape)
print("Fold 0 test:", example["X_test"].shape)


## 5. Re-run only the RBF-SVM search and generate validation probabilities

This is much smaller than rerunning the whole classical baseline because only the SVM is trained.

Per fold:

- 32 SVM configurations × 5 inner folds = 160 search fits;
- only the top 5 inner-CV candidates are checked on outer validation;
- the selected SVM is then fitted for seeds 17, 42, and 73;
- validation and test probabilities are saved.

The reproduced test predictions are compared against the downloaded original SVM predictions.


In [ ]:
INNER_CV = StratifiedKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=SEARCH_SEED,
)

SCORING = {
    "roc_auc": "roc_auc",
    "ap": "average_precision",
}

def top_candidates(
    cv_results,
    top_k=TOP_K_VALIDATION,
):
    df = pd.DataFrame(
        cv_results
    ).copy()

    return (
        df.sort_values(
            [
                "mean_test_roc_auc",
                "mean_test_ap",
            ],
            ascending=[False, False],
        )
        .head(top_k)
        .reset_index(drop=True)
    )

def validation_select(
    top_df,
    X_train,
    y_train,
    X_val,
    y_val,
):
    rows = []

    for rank, row in top_df.iterrows():
        params = dict(
            row["params"]
        )

        model = SVC(
            kernel="rbf",
            probability=False,
            cache_size=2000,
            **params,
        )

        model.fit(
            X_train,
            y_train,
        )

        scores = model.decision_function(
            X_val
        )

        rows.append({
            "inner_cv_rank": rank + 1,
            "inner_cv_roc_auc":
                float(row["mean_test_roc_auc"]),
            "inner_cv_ap":
                float(row["mean_test_ap"]),
            "validation_roc_auc":
                float(roc_auc_score(y_val, scores)),
            "validation_ap":
                float(average_precision_score(y_val, scores)),
            "params": params,
        })

    ranking = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "validation_roc_auc",
                "validation_ap",
                "inner_cv_roc_auc",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    return (
        dict(ranking.iloc[0]["params"]),
        ranking,
    )

svm_validation_frames = []
svm_reproduced_test_frames = []
svm_search_rows = []
reproduction_rows = []

for fold in FOLDS:
    print("\n" + "=" * 80)
    print("SVM FOLD", fold)
    print("=" * 80)

    M = make_outer_fold_matrices(
        fold
    )

    base = SVC(
        kernel="rbf",
        probability=False,
        cache_size=2000,
    )

    grid = {
        "C": SVM_C_VALUES,
        "gamma": SVM_GAMMA_VALUES,
        "class_weight": [
            None,
            "balanced",
        ],
    }

    search = GridSearchCV(
        base,
        grid,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        n_jobs=-1,
        error_score="raise",
    )

    search.fit(
        M["X_train"],
        M["y_train"],
    )

    top = top_candidates(
        search.cv_results_
    )

    selected_params, ranking = validation_select(
        top,
        M["X_train"],
        M["y_train"],
        M["X_val"],
        M["y_val"],
    )

    print("Selected parameters:", selected_params)
    print(
        "Validation selection AUC:",
        ranking.iloc[0]["validation_roc_auc"],
    )

    ranking_out = ranking.copy()
    ranking_out["params_json"] = (
        ranking_out["params"]
        .apply(
            lambda x: json.dumps(
                x,
                sort_keys=True,
            )
        )
    )

    ranking_out.drop(
        columns=["params"]
    ).to_csv(
        TABLES_DIR
        / f"fold_{fold}_svm_validation_selection.csv",
        index=False,
    )

    svm_search_rows.append({
        "FOLD": fold,
        "SELECTED_PARAMS":
            json.dumps(selected_params, sort_keys=True),
        "SELECTED_VALIDATION_AUC":
            float(ranking.iloc[0]["validation_roc_auc"]),
        "SELECTED_VALIDATION_AP":
            float(ranking.iloc[0]["validation_ap"]),
    })

    for seed in SEEDS:
        model = SVC(
            kernel="rbf",
            probability=True,
            cache_size=2000,
            random_state=seed,
            **selected_params,
        )

        model.fit(
            M["X_train"],
            M["y_train"],
        )

        p_val = model.predict_proba(
            M["X_val"]
        )[:, 1]

        p_test_reproduced = model.predict_proba(
            M["X_test"]
        )[:, 1]

        val_df = pd.DataFrame({
            "RID": M["rid_val"],
            "TARGET": M["y_val"],
            "FOLD": fold,
            "SEED": seed,
            "SVM_P": p_val,
        })

        test_df = pd.DataFrame({
            "RID": M["rid_test"],
            "TARGET": M["y_test"],
            "FOLD": fold,
            "SEED": seed,
            "SVM_P_REPRODUCED": p_test_reproduced,
        })

        svm_validation_frames.append(
            val_df
        )

        svm_reproduced_test_frames.append(
            test_df
        )

        original_path = (
            ORIGINAL_SVM_TEST_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )

        original = pd.read_csv(
            original_path
        )

        merged_check = test_df.merge(
            original[
                [
                    "RID",
                    "TARGET",
                    "PROB_PMCI",
                ]
            ],
            on=[
                "RID",
                "TARGET",
            ],
            how="inner",
            validate="one_to_one",
        )

        max_abs_diff = float(
            np.max(
                np.abs(
                    merged_check["SVM_P_REPRODUCED"]
                    - merged_check["PROB_PMCI"]
                )
            )
        )

        mean_abs_diff = float(
            np.mean(
                np.abs(
                    merged_check["SVM_P_REPRODUCED"]
                    - merged_check["PROB_PMCI"]
                )
            )
        )

        reproduction_rows.append({
            "FOLD": fold,
            "SEED": seed,
            "MAX_ABS_PROB_DIFF":
                max_abs_diff,
            "MEAN_ABS_PROB_DIFF":
                mean_abs_diff,
            "REPRODUCED_TEST_AUC":
                roc_auc_score(
                    merged_check["TARGET"],
                    merged_check["SVM_P_REPRODUCED"],
                ),
            "ORIGINAL_TEST_AUC":
                roc_auc_score(
                    merged_check["TARGET"],
                    merged_check["PROB_PMCI"],
                ),
        })

svm_validation_raw = pd.concat(
    svm_validation_frames,
    ignore_index=True,
)

svm_reproduced_test_raw = pd.concat(
    svm_reproduced_test_frames,
    ignore_index=True,
)

svm_search_summary = pd.DataFrame(
    svm_search_rows
)

reproduction_summary = pd.DataFrame(
    reproduction_rows
)

print("\nSelected SVM configurations:")
display(svm_search_summary)

print("\nReproduction check against downloaded original SVM test predictions:")
display(reproduction_summary)

print(
    "\nLargest probability difference across all reproduced test predictions:",
    reproduction_summary["MAX_ABS_PROB_DIFF"].max(),
)

svm_validation_raw.to_csv(
    PREDICTIONS_DIR
    / "svm_validation_predictions_all_folds_seeds.csv",
    index=False,
)

reproduction_summary.to_csv(
    TABLES_DIR
    / "svm_reproduction_check.csv",
    index=False,
)

svm_search_summary.to_csv(
    TABLES_DIR
    / "svm_selected_params_by_fold.csv",
    index=False,
)


## 6. Build 3-seed SVM and neural validation/test ensembles

For the **test set**, Uses the downloaded original SVM probabilities, not the reproduced copies, so the comparison remains anchored to the original ~0.9403 SVM baseline.


In [ ]:
def aggregate_svm_validation(fold):
    g = svm_validation_raw.loc[
        svm_validation_raw["FOLD"] == fold
    ]

    return (
        g.groupby("RID")
        .agg(
            TARGET=("TARGET", "first"),
            SVM_P=("SVM_P", "mean"),
            SVM_P_SD=("SVM_P", "std"),
        )
        .reset_index()
    )

def aggregate_original_svm_test(fold):
    frames = []

    for seed in SEEDS:
        path = (
            ORIGINAL_SVM_TEST_DIR
            / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
        )

        f = pd.read_csv(path)

        f = f[
            [
                "RID",
                "TARGET",
                "PROB_PMCI",
            ]
        ].copy()

        f["SEED"] = seed
        frames.append(f)

    raw = pd.concat(
        frames,
        ignore_index=True,
    )

    return (
        raw.groupby("RID")
        .agg(
            TARGET=("TARGET", "first"),
            SVM_P=("PROB_PMCI", "mean"),
            SVM_P_SD=("PROB_PMCI", "std"),
        )
        .reset_index()
    )

def aggregate_neural(fold, split):
    filename = (
        "best_validation_predictions.csv"
        if split == "validation"
        else "test_predictions.csv"
    )

    frames = []

    for seed in SEEDS:
        path = (
            NEURAL_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
            / "predictions"
            / filename
        )

        f = pd.read_csv(path)

        required = [
            "RID",
            "TARGET",
            "FINAL_P_pMCI",
            "TMC_P_pMCI",
        ]

        missing = [
            c for c in required
            if c not in f.columns
        ]

        if missing:
            raise KeyError(
                f"{path}: missing {missing}"
            )

        f = f.copy()
        f["SEED"] = seed
        frames.append(f)

    raw = pd.concat(
        frames,
        ignore_index=True,
    )

    return (
        raw.groupby("RID")
        .agg(
            TARGET=("TARGET", "first"),
            HYBRID_P=("FINAL_P_pMCI", "mean"),
            HYBRID_P_SD=("FINAL_P_pMCI", "std"),
            TMC_P=("TMC_P_pMCI", "mean"),
            TMC_P_SD=("TMC_P_pMCI", "std"),
        )
        .reset_index()
    )

validation_fold_tables = {}
test_fold_tables = {}

for fold in FOLDS:
    svm_val = aggregate_svm_validation(
        fold
    )

    svm_test = aggregate_original_svm_test(
        fold
    )

    neural_val = aggregate_neural(
        fold,
        "validation",
    )

    neural_test = aggregate_neural(
        fold,
        "test",
    )

    val = neural_val.merge(
        svm_val,
        on="RID",
        how="inner",
        suffixes=("_NEURAL", "_SVM"),
        validate="one_to_one",
    )

    test = neural_test.merge(
        svm_test,
        on="RID",
        how="inner",
        suffixes=("_NEURAL", "_SVM"),
        validate="one_to_one",
    )

    for table, name in [
        (val, "validation"),
        (test, "test"),
    ]:
        if not (
            table["TARGET_NEURAL"].astype(int)
            == table["TARGET_SVM"].astype(int)
        ).all():
            raise ValueError(
                f"Fold {fold}: target mismatch on {name}."
            )

        table.rename(
            columns={
                "TARGET_NEURAL": "TARGET"
            },
            inplace=True,
        )

        table.drop(
            columns=["TARGET_SVM"],
            inplace=True,
        )

        table["FOLD"] = fold

    validation_fold_tables[
        fold
    ] = val

    test_fold_tables[
        fold
    ] = test

print("Fold table sizes:")
for fold in FOLDS:
    print(
        fold,
        "validation=", len(validation_fold_tables[fold]),
        "test=", len(test_fold_tables[fold]),
    )


## 7. Select `alpha` on validation only

`alpha = 1` means pure SVM.  
`alpha = 0` means pure neural.

Tie-breaking is deterministic:

1. highest validation ROC AUC;
2. highest validation average precision;
3. closest to 0.5;
4. smaller alpha if still tied.


In [ ]:
def select_alpha(
    validation_df,
    neural_col,
):
    rows = []

    y = validation_df[
        "TARGET"
    ].to_numpy(dtype=int)

    svm_p = validation_df[
        "SVM_P"
    ].to_numpy(dtype=float)

    neural_p = validation_df[
        neural_col
    ].to_numpy(dtype=float)

    for alpha in ALPHA_GRID:
        p = (
            alpha * svm_p
            + (1.0 - alpha) * neural_p
        )

        rows.append({
            "ALPHA": float(alpha),
            "VALIDATION_ROC_AUC":
                float(
                    roc_auc_score(
                        y,
                        p,
                    )
                ),
            "VALIDATION_AP":
                float(
                    average_precision_score(
                        y,
                        p,
                    )
                ),
            "DISTANCE_FROM_0P5":
                abs(float(alpha) - 0.5),
        })

    ranking = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "VALIDATION_ROC_AUC",
                "VALIDATION_AP",
                "DISTANCE_FROM_0P5",
                "ALPHA",
            ],
            ascending=[
                False,
                False,
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return (
        float(ranking.iloc[0]["ALPHA"]),
        ranking,
    )

alpha_rows = []
alpha_rankings = {}

for fold in FOLDS:
    val = validation_fold_tables[
        fold
    ]

    alpha_hybrid, ranking_hybrid = select_alpha(
        val,
        "HYBRID_P",
    )

    alpha_tmc, ranking_tmc = select_alpha(
        val,
        "TMC_P",
    )

    alpha_rankings[
        (fold, "hybrid")
    ] = ranking_hybrid

    alpha_rankings[
        (fold, "tmc")
    ] = ranking_tmc

    alpha_rows.append({
        "FOLD": fold,

        "ALPHA_SVM_HYBRID":
            alpha_hybrid,

        "HYBRID_VALIDATION_FUSION_AUC":
            ranking_hybrid.iloc[0][
                "VALIDATION_ROC_AUC"
            ],

        "HYBRID_VALIDATION_SVM_AUC":
            roc_auc_score(
                val["TARGET"],
                val["SVM_P"],
            ),

        "HYBRID_VALIDATION_NEURAL_AUC":
            roc_auc_score(
                val["TARGET"],
                val["HYBRID_P"],
            ),

        "ALPHA_SVM_TMC":
            alpha_tmc,

        "TMC_VALIDATION_FUSION_AUC":
            ranking_tmc.iloc[0][
                "VALIDATION_ROC_AUC"
            ],

        "TMC_VALIDATION_SVM_AUC":
            roc_auc_score(
                val["TARGET"],
                val["SVM_P"],
            ),

        "TMC_VALIDATION_NEURAL_AUC":
            roc_auc_score(
                val["TARGET"],
                val["TMC_P"],
            ),
    })

    ranking_hybrid.to_csv(
        TABLES_DIR
        / f"fold_{fold}_alpha_ranking_svm_hybrid.csv",
        index=False,
    )

    ranking_tmc.to_csv(
        TABLES_DIR
        / f"fold_{fold}_alpha_ranking_svm_tmc.csv",
        index=False,
    )

alpha_summary = pd.DataFrame(
    alpha_rows
)

display(alpha_summary)

print("\nSelected Hybrid alphas:")
print(
    alpha_summary[
        "ALPHA_SVM_HYBRID"
    ].tolist()
)

print("\nSelected TMC alphas:")
print(
    alpha_summary[
        "ALPHA_SVM_TMC"
    ].tolist()
)

alpha_summary.to_csv(
    TABLES_DIR
    / "validation_selected_alpha_by_fold.csv",
    index=False,
)


## 8. Apply the locked alpha to each outer test fold


In [ ]:
test_outputs = []

for fold in FOLDS:
    test = test_fold_tables[
        fold
    ].copy()

    alpha_row = alpha_summary.loc[
        alpha_summary["FOLD"] == fold
    ].iloc[0]

    alpha_hybrid = float(
        alpha_row["ALPHA_SVM_HYBRID"]
    )

    alpha_tmc = float(
        alpha_row["ALPHA_SVM_TMC"]
    )

    test[
        "P_VALIDATION_SELECTED_SVM_HYBRID"
    ] = (
        alpha_hybrid
        * test["SVM_P"]
        + (1.0 - alpha_hybrid)
        * test["HYBRID_P"]
    )

    test[
        "P_VALIDATION_SELECTED_SVM_TMC"
    ] = (
        alpha_tmc
        * test["SVM_P"]
        + (1.0 - alpha_tmc)
        * test["TMC_P"]
    )

    # Keep the already-tested equal fusions as references.
    test[
        "P_EQUAL_SVM_HYBRID"
    ] = (
        0.5 * test["SVM_P"]
        + 0.5 * test["HYBRID_P"]
    )

    test[
        "P_EQUAL_SVM_TMC"
    ] = (
        0.5 * test["SVM_P"]
        + 0.5 * test["TMC_P"]
    )

    test[
        "ALPHA_SVM_HYBRID"
    ] = alpha_hybrid

    test[
        "ALPHA_SVM_TMC"
    ] = alpha_tmc

    test_outputs.append(
        test
    )

oof = pd.concat(
    test_outputs,
    ignore_index=True,
)

print(
    "OOF participants:",
    len(oof),
)

assert len(oof) == 544
assert oof["RID"].nunique() == 544

oof.to_csv(
    PREDICTIONS_DIR
    / "validation_selected_fusion_oof_predictions.csv",
    index=False,
)


## 9. Pooled performance


In [ ]:
def basic_metrics(
    y,
    p,
    threshold=0.5,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    p = np.asarray(
        p,
        dtype=float,
    )

    pred = (
        p >= threshold
    ).astype(int)

    tp = int(
        np.sum(
            (y == 1)
            & (pred == 1)
        )
    )

    tn = int(
        np.sum(
            (y == 0)
            & (pred == 0)
        )
    )

    fp = int(
        np.sum(
            (y == 0)
            & (pred == 1)
        )
    )

    fn = int(
        np.sum(
            (y == 1)
            & (pred == 0)
        )
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn)
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "ROC_AUC":
            roc_auc_score(
                y,
                p,
            ),

        "AP":
            average_precision_score(
                y,
                p,
            ),

        "SENSITIVITY":
            sensitivity,

        "SPECIFICITY":
            specificity,

        "BALANCED_ACCURACY":
            np.nanmean(
                [
                    sensitivity,
                    specificity,
                ]
            ),

        "BRIER":
            np.mean(
                (p - y) ** 2
            ),

        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
    }

methods = {
    "svm":
        "SVM_P",

    "medium_hybrid":
        "HYBRID_P",

    "medium_tmc":
        "TMC_P",

    "equal_svm_hybrid":
        "P_EQUAL_SVM_HYBRID",

    "equal_svm_tmc":
        "P_EQUAL_SVM_TMC",

    "validation_selected_svm_hybrid":
        "P_VALIDATION_SELECTED_SVM_HYBRID",

    "validation_selected_svm_tmc":
        "P_VALIDATION_SELECTED_SVM_TMC",
}

performance_rows = []

for method, col in methods.items():
    m = basic_metrics(
        oof["TARGET"],
        oof[col],
    )

    performance_rows.append({
        "METHOD": method,
        **m,
    })

performance = (
    pd.DataFrame(
        performance_rows
    )
    .sort_values(
        "ROC_AUC",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(performance)

performance.to_csv(
    TABLES_DIR
    / "pooled_performance.csv",
    index=False,
)


## 10. Fold-level performance and alpha stability


In [ ]:
fold_rows = []

for fold in FOLDS:
    g = oof.loc[
        oof["FOLD"] == fold
    ]

    for method, col in methods.items():
        m = basic_metrics(
            g["TARGET"],
            g[col],
        )

        fold_rows.append({
            "FOLD": fold,
            "METHOD": method,
            **m,
        })

fold_performance = pd.DataFrame(
    fold_rows
)

display(
    fold_performance.pivot(
        index="FOLD",
        columns="METHOD",
        values="ROC_AUC",
    )
)

fold_performance.to_csv(
    TABLES_DIR
    / "fold_performance.csv",
    index=False,
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    alpha_summary["FOLD"],
    alpha_summary["ALPHA_SVM_HYBRID"],
    marker="o",
    label="SVM + Hybrid alpha",
)

ax.plot(
    alpha_summary["FOLD"],
    alpha_summary["ALPHA_SVM_TMC"],
    marker="o",
    label="SVM + TMC alpha",
)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
)

ax.set_xlabel("Outer fold")
ax.set_ylabel("Validation-selected SVM weight (alpha)")
ax.set_title("Fusion-weight stability across folds")
ax.set_xticks(FOLDS)
ax.set_ylim(-0.02, 1.02)
ax.legend()
fig.tight_layout()

alpha_fig = (
    FIGURES_DIR
    / "validation_selected_alpha_by_fold.png"
)

fig.savefig(
    alpha_fig,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

print("Saved:", alpha_fig)


## 11. Paired participant bootstrap against SVM and equal fusion


In [ ]:
def paired_bootstrap_auc_difference(
    y,
    p_a,
    p_b,
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    p_a = np.asarray(
        p_a,
        dtype=float,
    )

    p_b = np.asarray(
        p_b,
        dtype=float,
    )

    idx0 = np.flatnonzero(
        y == 0
    )

    idx1 = np.flatnonzero(
        y == 1
    )

    rng = np.random.default_rng(
        seed
    )

    diffs = np.empty(
        n_bootstrap
    )

    for b in range(
        n_bootstrap
    ):
        s0 = rng.choice(
            idx0,
            size=len(idx0),
            replace=True,
        )

        s1 = rng.choice(
            idx1,
            size=len(idx1),
            replace=True,
        )

        idx = np.concatenate(
            [
                s0,
                s1,
            ]
        )

        diffs[b] = (
            roc_auc_score(
                y[idx],
                p_a[idx],
            )
            - roc_auc_score(
                y[idx],
                p_b[idx],
            )
        )

    observed = (
        roc_auc_score(
            y,
            p_a,
        )
        - roc_auc_score(
            y,
            p_b,
        )
    )

    return {
        "AUC_DIFF":
            observed,

        "CI_LOW":
            np.quantile(
                diffs,
                0.025,
            ),

        "CI_HIGH":
            np.quantile(
                diffs,
                0.975,
            ),

        "P_DIFF_GT_0":
            np.mean(
                diffs > 0
            ),
    }

comparisons = [
    (
        "validation_selected_svm_hybrid",
        "svm",
    ),
    (
        "validation_selected_svm_tmc",
        "svm",
    ),
    (
        "validation_selected_svm_hybrid",
        "equal_svm_hybrid",
    ),
    (
        "validation_selected_svm_tmc",
        "equal_svm_tmc",
    ),
    (
        "validation_selected_svm_hybrid",
        "medium_hybrid",
    ),
    (
        "validation_selected_svm_tmc",
        "medium_tmc",
    ),
]

bootstrap_rows = []

for i, (
    method_a,
    method_b,
) in enumerate(comparisons):

    result = paired_bootstrap_auc_difference(
        oof["TARGET"],
        oof[
            methods[method_a]
        ],
        oof[
            methods[method_b]
        ],
        seed=(
            BOOTSTRAP_SEED
            + i
        ),
    )

    bootstrap_rows.append({
        "METHOD_A":
            method_a,

        "METHOD_B":
            method_b,

        **result,
    })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES_DIR
    / "paired_bootstrap_auc_differences.csv",
    index=False,
)


## 12. Final interpretation block

The most useful things to inspect are:

- the five selected `alpha` values;
- whether they cluster toward SVM (`alpha > 0.5`) or vary wildly;
- whether validation-selected fusion beats the fixed 0.5 fusion;
- whether either fusion beats the original SVM pooled AUC;
- whether the paired bootstrap CI excludes zero.

A validation-selected result that is only numerically higher but has a wide interval crossing zero should be described as **suggestive**, not definitive.


In [ ]:
print("=" * 90)
print("VALIDATION-SELECTED SVM + NEURAL FUSION SUMMARY")
print("=" * 90)

print("\nSelected fusion weights:")
display(
    alpha_summary[
        [
            "FOLD",
            "ALPHA_SVM_HYBRID",
            "ALPHA_SVM_TMC",
        ]
    ]
)

print("\nPooled performance:")
display(
    performance[
        [
            "METHOD",
            "ROC_AUC",
            "AP",
            "BALANCED_ACCURACY",
            "SENSITIVITY",
            "SPECIFICITY",
            "BRIER",
        ]
    ]
)

print("\nPaired bootstrap comparisons:")
display(
    bootstrap_results
)

print("\nSVM reproduction check:")
display(
    reproduction_summary[
        [
            "FOLD",
            "SEED",
            "MAX_ABS_PROB_DIFF",
            "MEAN_ABS_PROB_DIFF",
            "REPRODUCED_TEST_AUC",
            "ORIGINAL_TEST_AUC",
        ]
    ]
)

print("\nSaved outputs:")
for p in sorted(
    TABLES_DIR.glob("*.csv")
):
    print(" -", p)

for p in sorted(
    PREDICTIONS_DIR.glob("*.csv")
):
    print(" -", p)

print(
    "\nInterpretation reminder: alpha was selected only on each outer fold's "
    "validation set. The outer-test labels were never used to choose alpha."
)
